In [1]:
import asyncio
import operator
import json
import random
from typing import Annotated, Sequence, TypedDict

from langchain_chroma import Chroma
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, BaseMessage, HumanMessage, ToolMessage
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langgraph.graph import StateGraph, END
from langgraph.prebuilt import tools_condition, ToolNode

/var/folders/qp/9vxvmncx0ks8cprx94py8fdh0000gn/T/ipykernel_87953/3158073463.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import AsyncHtmlLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.
/Users/aravindnatarajan/agents/lib/python3.14/site-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [2]:
EMBEDDING_MODEL = 'nomic-embed-text:latest'
LOCAL_LLM = 'gemma4:e4b'
TEMPERATURE = 0.7

In [3]:
embedding = OllamaEmbeddings(model=EMBEDDING_MODEL)
llm_model = ChatOllama(
    model=LOCAL_LLM,
    temperature=TEMPERATURE,
    use_responses_api=True
)

In [4]:
async def build_vectorstore(destinations: Sequence[str]) -> Chroma:
    urls = [f'https://en.wikivoyage.org/wiki/{destination}' for destination in destinations]
    loader = AsyncHtmlLoader(urls, default_parser="html.parser")
    print("Downloading destination pages ...")
    docs = await loader.aload()

    splitter = RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=128)
    chunks = sum([splitter.split_documents([d]) for d in docs], [])

    print(f"Embedding {len(chunks)} chunks ...")
    BATCH_SIZE = 100  # Safe batch size for Ollama requests
    
    vectordb_client = Chroma.from_documents(
        documents=chunks[:BATCH_SIZE],
        embedding=embedding,
    )
    
    for i in range(BATCH_SIZE, len(chunks), BATCH_SIZE):
        batch = chunks[i:i + BATCH_SIZE]
        print(f"Processing batch {i} to {min(i + BATCH_SIZE, len(chunks))}...")
        vectordb_client.add_documents(batch)
        
    print("Vector store ready.\n")
    return vectordb_client

In [5]:
UK_DESTINATIONS = [
    'Cornwall',
    'North_Cornwall',
    'South_Cornwall',
    'West_Cornwall',
    'Truro_(England)',
    'Newquay',
    'Port_Isaac',
    'St_Ives',
]

async def get_travel_info_vectorstore() -> Chroma:
    vectorstore_client = await build_vectorstore(UK_DESTINATIONS)
    return vectorstore_client

In [6]:
ti_vectorstore_client = await get_travel_info_vectorstore()
ti_retriever = ti_vectorstore_client.as_retriever()

Fetching pages: 100%|##################################################| 8/8 [00:01<00:00,  7.19it/s]


Embedding 2326 chunks ...
Processing batch 100 to 200...
Processing batch 200 to 300...
Processing batch 300 to 400...
Processing batch 400 to 500...
Processing batch 500 to 600...
Processing batch 600 to 700...
Processing batch 700 to 800...
Processing batch 800 to 900...
Processing batch 900 to 1000...
Processing batch 1000 to 1100...
Processing batch 1100 to 1200...
Processing batch 1200 to 1300...
Processing batch 1300 to 1400...
Processing batch 1400 to 1500...
Processing batch 1500 to 1600...
Processing batch 1600 to 1700...
Processing batch 1700 to 1800...
Processing batch 1800 to 1900...
Processing batch 1900 to 2000...
Processing batch 2000 to 2100...
Processing batch 2100 to 2200...
Processing batch 2200 to 2300...
Processing batch 2300 to 2326...
Vector store ready.



In [7]:
class WeatherForecast(TypedDict):
    town: str
    weather: Literal['sunny', 'foggy', 'rainy', 'windy']
    temperature: int
    
@tool(description='Get the weather forecast given the town name.')
def weather_forecast(town: str) -> dict:
    '''Get a weather forecast for a given town.
    Returns a WeatherForecast object with weather and temperature.
    '''
    _weather_options = ['sunny', 'foggy', 'rainy', 'windy']
    _temp_min = 18
    _temp_max = 31
    
    weather = random.choice(_weather_options)
    temperature = random.randint(_temp_min, _temp_max)
    return WeatherForecast(town=town, weather=weather, temperature=temperature)
    
@tool(description='Search travel information about destinations in England.')
def search_travel_info(query: str) -> str:
    """Search embedded WikiVoyage content for 
    information about destinations in England."""    
    docs = ti_retriever.invoke(query)
    top = docs[:4] if isinstance(docs, list) else docs
    return "\n---\n".join(d.page_content for d in top)    

In [8]:
tools = [weather_forecast, search_travel_info]
llm_with_tools = llm_model.bind_tools(tools)

In [9]:
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]

def llm_node(state: AgentState):
    """LLM node that decides whether to call tools."""
    # Build your instructions safely
    system_message = SystemMessage(content='''
    You are a helpful travel assistant that searches information and retrieves weather forecasts.
    
    CRITICAL RULES:
    1. Only suggest destinations that you have found inside the 'search_travel_info' tool.
    2. Identify candidate towns from your travel info search and check the weather for MULTIPLE candidate towns in parallel (simultaneously) to find the ones with the best weather.
    3. If your initial batch of towns has bad weather, query the weather for any backup towns in a single batch before formulating your final answer.
    ''')

    # Prepend the system prompt without mutating state["messages"]
    messages_for_llm = [system_message] + state["messages"]
    
    response_message = llm_with_tools.invoke(messages_for_llm)
    return {"messages": [response_message]}
    
tool_node = ToolNode(tools)

In [10]:
builder = StateGraph(AgentState)
builder.add_node("llm_node", llm_node)
builder.add_node("tools", tool_node)

builder.add_conditional_edges("llm_node", tools_condition)
builder.add_edge("tools", "llm_node")

builder.set_entry_point("llm_node")
travel_info_agent = builder.compile()

In [11]:
def chat_loop():
    print("UK Travel Assistant (type 'exit' to quit)")
    while True:
        user_input = input("You: ").strip()
        if user_input.lower() in {"exit", "quit"}:
            break
        state = {"messages": [HumanMessage(content=user_input)]}
        
        # --- FIX HERE: Add a recursion limit config ---
        # 10 steps is more than enough for a search + weather fallback batch
        config = {"recursion_limit": 10} 
        
        try:
            result = travel_info_agent.invoke(state, config=config)
            print("\n\n\n")
            print(result)
            print("\n\n\n")
            response_msg = result["messages"][-1].content
            print(f"Assistant: {response_msg}\n")
        except GraphRecursionError:
            # Catch the limit gracefully if it hits a runaway loop
            print("\nAssistant: I'm sorry, I couldn't find any towns with ideal weather after checking several options.\n")

            

In [12]:
chat_loop()

UK Travel Assistant (type 'exit' to quit)


You:  Suggest two Cornwall beach towns with nice weather.






{'messages': [HumanMessage(content='Suggest two Cornwall beach towns with nice weather.', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-06-09T18:12:44.953252Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6211151666, 'load_duration': 3372813708, 'prompt_eval_count': 228, 'prompt_eval_duration': 161232000, 'eval_count': 186, 'eval_duration': 2674055000, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'}, id='lc_run--019ead96-24d2-7c52-bae8-407e8a04ba2d-0', tool_calls=[{'name': 'search_travel_info', 'args': {'query': 'beach towns in Cornwall'}, 'id': 'e15c4ace-46f1-44f8-a179-8ebbbe7a133d', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 228, 'output_tokens': 186, 'total_tokens': 414}), ToolMessage(content='Cornwall.jpg|300px]]\\&quot;}}&quot;}}">3</a></span> <span id="Falmouth" class="fn org listing-name"><a rel="mw:W

You:  exit
